# Single-Asset Forecasting

This notebook compares two forecasting paths for a single tradable asset:

- A technical-indicator model trained on daily market features.
- A deeper transformer-style time-series model trained on rolling feature sequences.

Each path produces a next-day forecast and a next-month forecast (21 trading days).

In [4]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "src").exists() and (candidate / "backend").exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from src.market_forecasting import (
    AssetDefinition,
    describe_market_data_sources,
    describe_runtime_dependencies,
    is_deep_model_available,
    run_single_asset_experiment,
    summarize_experiment_results,
)

In [5]:
describe_runtime_dependencies()

,package,available,python_executable
0,numpy,True,/Users/tayebi/miniconda3/envs/all/bin/python
1,pandas,True,/Users/tayebi/miniconda3/envs/all/bin/python
2,scikit-learn,True,/Users/tayebi/miniconda3/envs/all/bin/python
3,torch,False,/Users/tayebi/miniconda3/envs/all/bin/python


In [6]:
ASSET = AssetDefinition(symbol="SIDU", label="Sidus Space")
START_DATE = "2022-01-01"
HORIZONS = (1, 21)
LOOKBACK = 60
EPOCHS = 20

print(f"Deep model available: {is_deep_model_available()}")
describe_market_data_sources(ASSET.symbol)

Deep model available: False


,source,available,details
0,backend_csv,False,/Users/tayebi/Documents/Repositories/GitHub/vo...
1,notebook_cache,False,/Users/tayebi/Documents/Repositories/GitHub/vo...
2,supabase_env,True,requires SUPABASE_URL and SUPABASE_SERVICE_KEY
3,twelvedata_env,True,requires TWELVE_DATA_API_KEY


In [7]:
# If the dependency table above shows `torch=False`, this cell will still run the technical model.
# Switch to the repo `.venv` kernel later if you want the deep model results too.
# If the market-data table shows no local CSV and no configured remote source,
# either add the asset through your TwelveData/Supabase pipeline or switch ASSET to one with local data.

results = run_single_asset_experiment(
    asset=ASSET,
    start_date=START_DATE,
    horizons=HORIZONS,
    lookback=LOOKBACK,
    epochs=EPOCHS,
)

summary = summarize_experiment_results(results)
summary

,horizon_days,model,rmse,mae,directional_accuracy,correlation,latest_signal,latest_predicted_return,latest_predicted_price
0,1,technical,0.121742,0.081108,0.485714,-0.025991,SHORT,-0.025616,4.141132
1,21,technical,0.525035,0.426853,0.592233,0.375289,SHORT,-0.241480,3.223710


In [8]:
{horizon: payload["deep_error"] for horizon, payload in results.items() if payload.get("deep_error")}

{1: 'The deep time-series model requires the `torch` package, but it is not installed in the active Python environment: /Users/tayebi/miniconda3/envs/all/bin/python. If you are running this notebook in Jupyter, switch to the repository `.venv` kernel or install PyTorch into the current kernel.',
 21: 'The deep time-series model requires the `torch` package, but it is not installed in the active Python environment: /Users/tayebi/miniconda3/envs/all/bin/python. If you are running this notebook in Jupyter, switch to the repository `.venv` kernel or install PyTorch into the current kernel.'}

In [9]:
latest_forecasts = pd.DataFrame(
    [
        {"horizon_days": horizon, "model": "technical", **payload["technical_forecast"]}
        for horizon, payload in results.items()
    ]
    + [
        {"horizon_days": horizon, "model": "deep", **payload["deep_forecast"]}
        for horizon, payload in results.items()
        if payload.get("deep_forecast") is not None
    ]
)
latest_forecasts.sort_values(["horizon_days", "model"]).reset_index(drop=True)

,horizon_days,model,as_of_date,target_date,predicted_log_return,predicted_simple_return,predicted_price,signal
0,1,technical,2026-04-10,2026-04-13,-0.025950,-0.025616,4.141132,SHORT
1,21,technical,2026-04-10,2026-05-11,-0.276386,-0.241480,3.223710,SHORT


In [10]:
results[1]["feature_importance"].head(15).to_frame("importance")

,importance
sidu_log_price,0.132851
sidu_log_return_21,0.063701
sidu_month,0.063447
sidu_sma_gap_10,0.050830
sidu_log_return_5,0.049529
sidu_macd_hist,0.046601
sidu_log_return_1,0.046155
sidu_sma_gap_21,0.040709
sidu_open_close_gap,0.039458
sidu_sma_gap_5,0.038688


In [11]:
daily_backtest_tail = results[1]["backtest"].tail(10)
monthly_backtest_tail = results[21]["backtest"].tail(10)

daily_backtest_tail, monthly_backtest_tail

(            actual_log_return  technical_pred_log_return  current_close  \
 Date                                                                      
 2026-03-26          -0.173472                  -0.018998           2.70   
 2026-03-27          -0.054312                  -0.021195           2.27   
 2026-03-30           0.076099                  -0.026532           2.15   
 2026-03-31          -0.099630                  -0.010261           2.32   
 2026-04-01           0.386234                  -0.011333           2.10   
 2026-04-02           0.172021                  -0.016242           3.09   
 2026-04-06           0.032174                  -0.006673           3.67   
 2026-04-07          -0.040382                  -0.037595           3.79   
 2026-04-08          -0.086013                  -0.041116           3.64   
 2026-04-09           0.240948                  -0.008368           3.34   
 
             actual_price  technical_pred_price  
 Date                               